In [1]:
# imports
# El dataset flows.csv se genera en 'Generacion_Flujos_Mirai.ipynb';
# este notebook solo lo consume (igual que el resto de notebooks Mirai).
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import KNeighborsClassifier
from sklearn.covariance import EllipticEnvelope
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, roc_auc_score
import warnings; warnings.filterwarnings('ignore')


[1/80] non-attack1.pcapng  (Mirai_CnC)
    781 flujos
[2/80] non-attack2.pcapng  (Mirai_CnC)
    698 flujos
[3/80] non-attack3.pcapng  (Mirai_CnC)
    717 flujos
[4/80] non-attack4.pcapng  (Mirai_CnC)
    716 flujos
[5/80] non-attack5.pcapng  (Mirai_CnC)
    868 flujos
[6/80] non-attack6.pcapng  (Mirai_CnC)
    873 flujos
[7/80] non-attack7.pcapng  (Mirai_CnC)
    859 flujos
[8/80] non-attack8.pcapng  (Mirai_CnC)
    712 flujos
[9/80] non-attack9.pcapng  (Mirai_CnC)
    832 flujos
[10/80] non-attack10.pcapng  (Mirai_CnC)
    846 flujos
[11/80] non-attack11.pcapng  (Mirai_CnC)
    841 flujos
[12/80] non-attack12.pcapng  (Mirai_CnC)
    847 flujos
[13/80] non-attack13.pcapng  (Mirai_CnC)
    784 flujos
[14/80] non-attack14.pcapng  (Mirai_CnC)
    805 flujos
[15/80] non-attack15.pcapng  (Mirai_CnC)
    809 flujos
[16/80] non-attack16.pcapng  (Mirai_CnC)
    918 flujos
[17/80] non-attack17.pcapng  (Mirai_CnC)
    899 flujos
[18/80] non-attack18.pcapng  (Mirai_CnC)
    929 flujos
[19/80] no

In [2]:
# preprocesamiento (igual que Autoencoder Mirai)

df = pd.read_csv(str(Path.cwd().parent / "data" / "flows.csv"))
df['src_ip'] = df['src_ip'].astype(str)

y_raw = df['label'].values

# Estado Markoviano reclasificado semánticamente
def reclassify_state(row):
    proto  = row['state']
    b_pkts = row['b_pkts']
    avg_ps = row['avg_pkt_size']
    if proto == 'DNS':
        return 'DNS_FLOOD'
    if proto in ('HTTP', 'HTTPS'):
        return 'HTTP_FLOOD'
    if proto == 'SSH':
        return 'OTHER'
    if proto == 'UDP_OTHER':
        if b_pkts == 0 and avg_ps < 100:
            return 'UDP_SMALL_NORESPONSE'
        elif b_pkts == 0:
            return 'UDP_LARGE_NORESPONSE'
        else:
            return 'UDP_BIDIRECTIONAL'
    if proto == 'TCP_OTHER':
        if b_pkts == 0 and avg_ps < 80:
            return 'TCP_SYN_LIKE'
        elif b_pkts == 0:
            return 'TCP_ACK_LIKE'
        else:
            return 'TCP_ESTABLISHED'
    return 'OTHER'

df['state'] = df.apply(reclassify_state, axis=1)

# Subsampleo por clase — mantiene tamaños manejables (basado en Hwang Tabla 12)
HWANG_CLASS_TOTAL = {
    'Normal':       68200 + 8525,
    'ACK_Flood':     6600 +  825,
    'DNS_Flood':     4312 +  539,
    'Mirai_CnC':    68200 + 8525,
    'GREIP_Flood':  24712 + 3089,
    'HTTP_Flood':     120 +   15,
    'SYN_Flood':    68200 + 8525,
    'UDP_Flood':    28816 + 3062,
    'VSE_Flood':     4432 +  554,
}

np.random.seed(42)
idx_keep_list = []
for cls_name, n_total in HWANG_CLASS_TOTAL.items():
    idx_cls   = np.where(df['class_name'] == cls_name)[0]
    available = len(idx_cls)
    if available == 0:
        print(f'  AVISO: clase {cls_name!r} no encontrada en el CSV')
        continue
    n_use = min(available, n_total)
    if available > n_total:
        idx_cls = np.random.choice(idx_cls, n_total, replace=False)
    idx_keep_list.append(idx_cls)
    print(f'  {cls_name:<14}: disponible={available:>7,}  objetivo={n_total:>7,}  usado={n_use:>7,}')

idx_keep = np.sort(np.concatenate(idx_keep_list))
df_sub   = df.iloc[idx_keep].reset_index(drop=True)
y        = y_raw[idx_keep]

# df_seq con estados en string (para secuencias Markovianas)
df_seq            = df_sub[['src_ip', 'flow_start', 'state']].copy()
df_seq['orig_idx'] = np.arange(len(df_sub))

# Features numéricas + LabelEncoder para state (igual que Autoencoder Mirai)
feature_cols = ['n_pkts', 'n_bytes', 'f_pkts', 'f_bytes',
                'b_pkts', 'b_bytes', 'avg_pkt_size', 'duration']
state_enc    = LabelEncoder().fit_transform(df_sub['state'].values)

X_vals    = np.column_stack([df_sub[feature_cols].values, state_enc]).astype(np.float32)
FEAT_COLS = feature_cols + ['state']
X_full    = pd.DataFrame(X_vals, columns=FEAT_COLS)

print(f'\nFeatures usadas ({len(FEAT_COLS)}): {FEAT_COLS}')
print(f'Normal (0): {(y==0).sum():,}  |  Ataque (1): {(y==1).sum():,}')
print(f'Anomalías : {y.mean():.1%}')
print(f'Total filas: {len(y):,}')
print(f'Estados únicos: {df_seq["state"].nunique()}')


  Normal        : disponible=  7,793  objetivo= 76,725  usado=  7,793
  ACK_Flood     : disponible=131,249  objetivo=  7,425  usado=  7,425
  DNS_Flood     : disponible=307,113  objetivo=  4,851  usado=  4,851
  Mirai_CnC     : disponible= 25,070  objetivo= 76,725  usado= 25,070
  GREIP_Flood   : disponible=  2,521  objetivo= 27,801  usado=  2,521
  HTTP_Flood    : disponible=  2,693  objetivo=    135  usado=    135
  SYN_Flood     : disponible=169,685  objetivo= 76,725  usado= 76,725
  UDP_Flood     : disponible=152,711  objetivo= 31,878  usado= 31,878
  VSE_Flood     : disponible=207,941  objetivo=  4,986  usado=  4,986

Features usadas (9): ['n_pkts', 'n_bytes', 'f_pkts', 'f_bytes', 'b_pkts', 'b_bytes', 'avg_pkt_size', 'duration', 'state']
Normal (0): 7,793  |  Ataque (1): 153,591
Anomalías : 95.2%
Total filas: 161,384
Estados únicos: 9


In [3]:
# definición y Entrenamiento

def build_tm(sequences, order):
    counts = defaultdict(lambda: defaultdict(int))
    for seq in sequences:
        for t in range(order, len(seq)):
            counts[tuple(seq[t - order:t])][seq[t]] += 1
    return {ctx: {s: n / sum(v.values()) for s, n in v.items()}
            for ctx, v in counts.items()}

def compute_scores(sequences, tm, order, eps=1e-10):
    out = []
    for seq in sequences:
        for t, state in enumerate(seq):
            if t < order:
                out.append((0., 0., 0.)); continue
            ctx   = tuple(seq[t - order:t])
            trans = tm.get(ctx, {})
            p     = trans.get(state, eps)
            dist  = np.array(list(trans.values())) if trans else np.array([eps])
            dist /= dist.sum()
            out.append((
                -np.log(p + eps),
                -np.sum(dist * np.log(dist + eps)),
                np.log(p + eps)
            ))
    return np.array(out)

def markov_features(df_seq, idx_tr, idx_ev, order):
    def get_seqs(idx):
        d = df_seq.iloc[idx].sort_values(['src_ip', 'flow_start'])
        g = d.groupby('src_ip', sort=False)
        return [list(x['state']) for _, x in g], [list(x['orig_idx']) for _, x in g]

    seqs_tr, _          = get_seqs(idx_tr)
    tm                  = build_tm(seqs_tr, order)
    seqs_ev, orig_idxs  = get_seqs(idx_ev)
    scores              = compute_scores(seqs_ev, tm, order)
    flat_orig           = [i for grp in orig_idxs for i in grp]
    reorder             = np.argsort(flat_orig)
    s = scores[reorder]
    return pd.DataFrame({'surprise': s[:, 0], 'entropy': s[:, 1], 'loglik': s[:, 2]})

def youden_thr(y_true, sc):
    fpr, tpr, thr = roc_curve(y_true, sc)
    return thr[np.argmax(tpr - fpr)]

def get_models(seed):
    return {
        'RandomForest'   : RandomForestClassifier(n_estimators=50, random_state=seed, n_jobs=-1),
        'MLP'            : MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=50,
                                         early_stopping=True, n_iter_no_change=5,
                                         tol=1e-3, random_state=seed),
        'KNN'            : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'IsolationForest': IsolationForest(n_estimators=100, contamination='auto',
                                           random_state=seed, n_jobs=-1),
        'OCSVM'          : OneClassSVM(kernel='rbf', nu=0.1, gamma='scale'),
        'RobustCov'      : EllipticEnvelope(contamination=0.3, random_state=seed),
    }

UNSUP_SUB     = {'OCSVM', 'RobustCov'}
SEEDS         = list(range(10))
MARKOV_ORDERS = [0, 1, 2, 3, 4]

all_results = []
t0_total = time.time()

for seed in SEEDS:
    t0_seed = time.time()

    # Split estratificado 64 / 16 / 20 (test=20%, igual que Autoencoder Mirai)
    X_tv, X_te, y_tv, y_te, idx_tv, idx_te = train_test_split(
        X_full, y, np.arange(len(y)), test_size=0.20, random_state=seed, stratify=y
    )
    X_tr, X_vl, y_tr, y_vl, idx_tr, idx_vl = train_test_split(
        X_tv, y_tv, idx_tv, test_size=0.20, random_state=seed, stratify=y_tv
    )

    # MinMaxScaler ajustado sobre train (post-split), igual que Autoencoder Mirai
    scaler  = MinMaxScaler()
    X_tr_s  = pd.DataFrame(scaler.fit_transform(X_tr), columns=FEAT_COLS)
    X_vl_s  = pd.DataFrame(scaler.transform(X_vl),     columns=FEAT_COLS)
    X_te_s  = pd.DataFrame(scaler.transform(X_te),     columns=FEAT_COLS)

    for k in MARKOV_ORDERS:
        t0_k = time.time()

        if k == 0:
            Xtr, Xvl, Xte = X_tr_s, X_vl_s, X_te_s
        else:
            mk_tr = markov_features(df_seq, idx_tr, idx_tr, k)
            mk_vl = markov_features(df_seq, idx_tr, idx_vl, k)
            mk_te = markov_features(df_seq, idx_tr, idx_te, k)
            Xtr = pd.concat([X_tr_s.reset_index(drop=True), mk_tr], axis=1)
            Xvl = pd.concat([X_vl_s.reset_index(drop=True), mk_vl], axis=1)
            Xte = pd.concat([X_te_s.reset_index(drop=True), mk_te], axis=1)

        Xtr_norm = Xtr[y_tr == 0]
        rng      = np.random.default_rng(seed)
        sub_idx  = rng.choice(len(Xtr_norm), size=min(5000, len(Xtr_norm)), replace=False)
        Xtr_sub  = Xtr_norm.iloc[sub_idx]

        for name, model in get_models(seed).items():
            t0_m = time.time()

            if name in ('RandomForest', 'MLP', 'KNN'):
                model.fit(Xtr, y_tr)
                sc_vl = model.predict_proba(Xvl)[:, 1]
                sc_te = model.predict_proba(Xte)[:, 1]
            else:
                fit_data = Xtr_sub if name in UNSUP_SUB else Xtr_norm
                model.fit(fit_data)
                score_fn = model.score_samples if hasattr(model, 'score_samples') else model.decision_function
                sc_vl = -score_fn(Xvl)
                sc_te = -score_fn(Xte)

            pred = (sc_te >= youden_thr(y_vl, sc_vl)).astype(int)
            all_results.append({
                'seed' : seed,
                'k'    : k,
                'model': name,
                'AUC'  : roc_auc_score(y_te, sc_te),
                'F1'   : f1_score(y_te, pred, zero_division=0),
                'Prec' : precision_score(y_te, pred, zero_division=0),
                'Rec'  : recall_score(y_te, pred, zero_division=0),
            })
            print(f'  seed={seed} k={k} {name:<18} {time.time()-t0_m:5.1f}s', flush=True)

        print(f'  --- k={k} completado en {time.time()-t0_k:.1f}s ---', flush=True)

    elapsed   = time.time() - t0_seed
    remaining = elapsed * (len(SEEDS) - seed - 1)
    print(f'Semilla {seed} completada en {elapsed:.0f}s | ETA restante: {remaining/60:.1f} min\n', flush=True)


  seed=0 k=0 RandomForest         0.5s
  seed=0 k=0 MLP                 20.3s
  seed=0 k=0 KNN                 17.5s
  seed=0 k=0 IsolationForest      0.4s
  seed=0 k=0 OCSVM                1.7s
  seed=0 k=0 RobustCov            0.4s
  --- k=0 completado en 40.8s ---
  seed=0 k=1 RandomForest         0.6s
  seed=0 k=1 MLP                  6.0s
  seed=0 k=1 KNN                  9.6s
  seed=0 k=1 IsolationForest      0.3s
  seed=0 k=1 OCSVM                1.7s
  seed=0 k=1 RobustCov            0.4s
  --- k=1 completado en 20.4s ---
  seed=0 k=2 RandomForest         0.5s
  seed=0 k=2 MLP                  6.0s
  seed=0 k=2 KNN                  5.2s
  seed=0 k=2 IsolationForest      0.3s
  seed=0 k=2 OCSVM                1.7s
  seed=0 k=2 RobustCov            0.5s
  --- k=2 completado en 16.0s ---
  seed=0 k=3 RandomForest         0.5s
  seed=0 k=3 MLP                  4.9s
  seed=0 k=3 KNN                  5.6s
  seed=0 k=3 IsolationForest      0.3s
  seed=0 k=3 OCSVM                1.6s
 

In [4]:
# resultados: métricas sobre el set de test

df_res = pd.DataFrame(all_results)
names  = list(df_res['model'].unique())

W     = 102
TITLE = 'TABLA DE RESULTADOS — AUC y F1-Score por modelo y orden Markoviano'
HDR   = (f"{'Modelo':<18} {'Orden':>5}   {'AUC media':>10}  {'±std':>6}   "
         f"{'F1 media':>9}  {'±std':>6}   {'Precisión':>9}  {'Recall':>7}   Tend.")
SEP   = '-' * W

print('=' * W)
print(TITLE.center(W))
print('=' * W)
print(HDR)
print(SEP)

for i, name in enumerate(names):
    aucs = [df_res[(df_res['model'] == name) & (df_res['k'] == k)]['AUC'].mean()
            for k in MARKOV_ORDERS]
    d = aucs[-1] - aucs[0]

    if   max(aucs) - min(aucs) < 0.001:                              tend = 'estable'
    elif all(aucs[j] >= aucs[j-1] for j in range(1, len(aucs))):     tend = f'mejora  d=+{d:.4f}'
    elif all(aucs[j] <= aucs[j-1] for j in range(1, len(aucs))):     tend = f'degrada d={d:.4f}'
    else:                                                             tend = f'mixta   d={d:+.4f}'

    for k in MARKOV_ORDERS:
        sub  = df_res[(df_res['model'] == name) & (df_res['k'] == k)]
        r    = {m: sub[m].mean() for m in ('AUC', 'F1', 'Prec', 'Rec')}
        rs   = {m: sub[m].std()  for m in ('AUC', 'F1')}

        if   k == 0:                            arrow = '-'
        elif aucs[k] > aucs[k-1] + 0.0005:     arrow = 'sube'
        elif aucs[k] < aucs[k-1] - 0.0005:     arrow = 'baja'
        else:                                   arrow = '='

        tend_col = tend if k == 0 else ''
        print(f"{name:<18} {'k='+str(k):>5}   {r['AUC']:>10.4f}  {rs['AUC']:>6.4f}   "
              f"{r['F1']:>9.4f}  {rs['F1']:>6.4f}   "
              f"{r['Prec']:>9.4f}  {r['Rec']:>7.4f}   {arrow:<5}  {tend_col}")

    if i < len(names) - 1:
        print()
        print(SEP)

print('=' * W)

print()
print('=' * 72)
print(f"{'RESUMEN DE TENDENCIAS Y VEREDICTO':^72}")
print('=' * 72)
for name in names:
    aucs   = [df_res[(df_res['model'] == name) & (df_res['k'] == k)]['AUC'].mean()
              for k in MARKOV_ORDERS]
    f1s    = [df_res[(df_res['model'] == name) & (df_res['k'] == k)]['F1'].mean()
              for k in MARKOV_ORDERS]
    best_k = int(np.argmax(aucs))
    d_auc  = aucs[-1] - aucs[0]
    print(f'\n  {name}')
    print('    AUC por orden: ' + '  '.join(f'k={k}:{aucs[k]:.4f}' for k in MARKOV_ORDERS))
    print('    F1  por orden: ' + '  '.join(f'k={k}:{f1s[k]:.4f}'  for k in MARKOV_ORDERS))
    print(f'    Mejor k       : k={best_k}  AUC={aucs[best_k]:.4f}  F1={f1s[best_k]:.4f}')
    if   d_auc > 0.002:   verdict = 'Markov AYUDA'
    elif d_auc < -0.002:  verdict = 'Markov PERJUDICA'
    else:                 verdict = 'Markov SIN EFECTO CLARO'
    print(f'    Veredicto     : {verdict}  (delta AUC k0→k4 = {d_auc:+.4f})')


                  TABLA DE RESULTADOS — AUC y F1-Score por modelo y orden Markoviano                  
Modelo             Orden    AUC media    ±std    F1 media    ±std   Precisión   Recall   Tend.
------------------------------------------------------------------------------------------------------
RandomForest         k=0       0.9991  0.0006      0.9985  0.0002      0.9998   0.9973   -      estable
RandomForest         k=1       0.9991  0.0005      0.9982  0.0003      0.9999   0.9965   =      
RandomForest         k=2       0.9994  0.0003      0.9983  0.0002      0.9999   0.9967   =      
RandomForest         k=3       0.9993  0.0003      0.9983  0.0003      0.9999   0.9968   =      
RandomForest         k=4       0.9993  0.0004      0.9983  0.0003      0.9999   0.9968   =      

------------------------------------------------------------------------------------------------------
MLP                  k=0       0.9976  0.0008      0.9978  0.0005      0.9994   0.9962   -      degrada